In [ ]:
import subprocess 
import os

import scienceplots 

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

import torch
import torch.nn.functional as F
import numpy as np
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from scipy.spatial.distance import cdist
from dotenv import load_dotenv
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from hydra.initialize import initialize_config_dir
from hydra.utils import instantiate

from flowers.models.rgbmnist import WrappedModel
from flow_matching.solver import ODESolver

load_dotenv()

DATA_ROOT = os.getenv("DATA_ROOT")

In [2]:
# Note: change experiment name and model number for your training run.

data_files = {
    "test": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0/test/*.parquet",
    "train": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0/train/*.parquet",
}
w1 = load_dataset(
    "parquet",
    data_files=data_files
)
keys = ["vae", "uncond", "cond"]
embeddings = {k: np.load(f"./tsne_embeddings/tsne_result_{k}.npy") for k in keys}
embeddings_test = {k: np.load(f"./tsne_embeddings/tsne_result_test_{k}.npy") for k in keys}

In [3]:
# Define the relative path to your config directory
config_dir_relative = "../../src/conf"
config_dir_absolute = os.path.join(os.getcwd(), config_dir_relative)
GlobalHydra.instance().clear()

with initialize_config_dir(config_dir=config_dir_absolute, version_base=None):
    cfg = compose(
        config_name="experiment/rgbmnist_Flow_v2/embed",
        overrides=[
            "++seed=42",
            f"+paths.embedding_dir={DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0",
            f"+lightning_loader.ckpt_path={DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/ckpts/7518770_0.ckpt",
            f"++loader.shuffle=False",
        ],
    )

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
raw_dataloader = instantiate(cfg.data.loader)
raw_dataloader.setup()
lightning_loader = instantiate(cfg.lightning_loader)
lightning_loader.to(device)

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from skimage.measure import moments, moments_central, inertia_tensor_eigvals
from tqdm import tqdm

z_vae = w1["train"]["orig"]

def compute_rotation_from_latents(z_vae, model, batch_size=128):
    # Ensure sequential alignment with the t-SNE coordinate indices
    if not isinstance(z_vae, torch.Tensor):
        z_vae = torch.tensor(z_vae)
    
    dataset = TensorDataset(z_vae)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    model.eval()
    results = []

    print("Reconstructing images and measuring geometry...")
    with torch.no_grad():
        for batch in tqdm(loader):
            latent_batch = batch[0].to(next(model.parameters()).device)
            
            # Reconstruction
            z_up = model.projection_up(latent_batch)
            reconstructed_imgs = model.decoder(z_up).cpu().numpy()
            
            for i in range(len(reconstructed_imgs)):
                # Take the first channel (robust for grayscale or RGB-MNIST)
                img = reconstructed_imgs[i, 0] if reconstructed_imgs.ndim == 4 else reconstructed_imgs[i]
                
                # Treat pixels as a distribution to find the principal axis
                M = moments(img, order=1)
                mass = M[0, 0]
                
                if mass <= 0:
                    results.append({"Rotation_Deg": 0, "Eccentricity": 0, "Major_L1": 0})
                    continue
                
                # Physics-based central moments
                centroid = (M[1, 0] / mass, M[0, 1] / mass)
                mu = moments_central(img, center=centroid, order=2)
                
                # Eigenvalues of the inertia tensor
                evals = inertia_tensor_eigvals(img, mu=mu)
                l1, l2 = evals[0], evals[1]
                
                # Calculate the orientation angle theta
                # Formula: theta = 0.5 * atan2(2 * mu_11, mu_20 - mu_02)
                angle = 0.5 * np.degrees(np.arctan2(2 * mu[1, 1], mu[2, 0] - mu[0, 2]))
                
                # Geometric Eccentricity (0 = circle, 1 = line)
                ecc = np.sqrt(1 - (l2 / (l1 + 1e-6)))
                
                results.append({
                    "Rotation_Deg": round(angle, 2),
                    "Eccentricity": round(ecc, 4),
                    "Major_L1": round(l1, 2)
                })

    return pd.DataFrame(results)

# Execute and synchronize with your existing embeddings
train_rotation_df = compute_rotation_from_latents(z_vae, lightning_loader.base_model)
train_rotation_df.to_csv("vae_train_rotation_aligned.csv", index=False)

In [17]:
wrapped_vf = WrappedModel(velocity_model=lightning_loader.vf)
solver = ODESolver(velocity_model=wrapped_vf)

In [23]:

# --- 1. Global Plotting Style ---
plt.style.use("dark_background")
plt.rcParams.update({
    "font.family": "serif",
    "figure.dpi": 200
})

def plot_tsne_manifold(embeddings_2d, base_latents, w1_data, 
                                target_digits=[0,1,2,3,4,5,6,7,8,9], 
                                min_dist=0.3, max_images=600, zoom=0.4):
    """
    Plots a filtered subset of digits at t-SNE coordinates with transparency.
    """
    fig, ax = plt.subplots(figsize=(14, 11))
    
    # --- STEP 1: Filter by Digit ---
    all_digits = np.array(w1_data["digit"])
    # Get indices where the digit is in our target list (e.g., [1, 4, 7])
    mask = np.isin(all_digits, target_digits)
    valid_indices = np.where(mask)[0]
    
    # Shuffle to ensure we get a random variety of the target digits
    np.random.shuffle(valid_indices)
    
    # --- STEP 2: Selection & Decluttering ---
    selected_indices = []
    coords_pool = []

    for i in valid_indices:
        curr_coords = embeddings_2d[i]
        if len(coords_pool) > 0:
            if np.min(cdist([curr_coords], coords_pool)) < min_dist:
                continue
        
        coords_pool.append(curr_coords)
        selected_indices.append(i)
        if len(selected_indices) >= max_images:
            break

    print(f"Plotting {len(selected_indices)} transparent images for digits: {target_digits}")

    # --- STEP 3: Construct Conditioning (y) ---
    r = torch.as_tensor(w1_data["r"])[selected_indices].float().reshape(-1, 1)
    g = torch.as_tensor(w1_data["g"])[selected_indices].float().reshape(-1, 1)
    
    digits_tensor = torch.as_tensor(w1_data["digit"])[selected_indices].long().reshape(-1)
    one_hot_digits = F.one_hot(digits_tensor, num_classes=10).float()
    
    # Create y: [r, g, one_hot] -> Shape [Batch, 12]
    y_cond = torch.cat([r, g, one_hot_digits], dim=1).to(device)
    
    # --- STEP 4: Run Flow Solver & Decoder ---
    z_init = torch.as_tensor(base_latents)[selected_indices].float().to(device)
    
    with torch.no_grad():
        sol = solver.sample(
            time_grid=torch.linspace(0, 1, 50).to(device),
            x_init=z_init,
            y=y_cond,
            step_size=0.005,
            return_intermediates=True,
        )
        z_final = sol[-1]
        
        z_up = lightning_loader.base_model.projection_up(z_final)
        images = lightning_loader.base_model.decoder(z_up).cpu()

    # --- STEP 5: Transparency & Plotting ---
    for idx_in_batch, original_idx in enumerate(selected_indices):
        # RGB: (H, W, 3)
        img_np = images[idx_in_batch].permute(1, 2, 0).numpy()
        
        # Normalize
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        img_np = np.clip(img_np, 0, 1)

        # Alpha Channel (intensity-based)
        alpha = np.max(img_np, axis=-1)
        img_rgba = np.dstack((img_np, alpha))

        coords = embeddings_2d[original_idx]
        imagebox = OffsetImage(img_rgba, zoom=zoom)
        ab = AnnotationBbox(imagebox, coords, frameon=False, pad=0.0)
        ax.add_artist(ab)

    # --- Styling ---
    ax.set_xlim(embeddings_2d[:, 0].min() - 5, embeddings_2d[:, 0].max() + 5)
    ax.set_ylim(embeddings_2d[:, 1].min() - 5, embeddings_2d[:, 1].max() + 5)
    ax.axis('off')
    
    plt.show()

# Conditional manifold (all digits)

In [ ]:
plot_tsne_manifold(
    embeddings_2d=embeddings["cond"], 
    base_latents=w1["train"]["cond"], 
    w1_data=w1["train"], 
    min_dist=0.3, 
    max_images=500, 
    zoom=0.5
)

# Conditional Manifold (1, 7)

In [ ]:
plot_tsne_manifold(
    embeddings_2d=embeddings["cond"], 
    base_latents=w1["train"]["cond"], 
    w1_data=w1["train"], 
    target_digits=[1, 7], 
    min_dist=0.3, 
    max_images=500, 
    zoom=0.5
)

# Conditional Manifold (0,9)

In [ ]:
plot_tsne_manifold(
    embeddings_2d=embeddings["cond"], 
    base_latents=w1["train"]["cond"], 
    w1_data=w1["train"], 
    target_digits=[0, 9], 
    min_dist=0.3, 
    max_images=500, 
    zoom=0.5
)